In [1]:
import pandas as pd

# How much does an average customer pay per month?
df_raw = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
avg_monthly_charge = df_raw['MonthlyCharges'].mean()

print("Average monthly charge: $", round(avg_monthly_charge, 2))

Average monthly charge: $ 64.76


In [2]:
retention_value_months = 12  # assumption: a retained customer stays ~12 more months
customer_value_at_risk = avg_monthly_charge * retention_value_months

print("Estimated value at risk per churned customer: $", round(customer_value_at_risk, 2))

Estimated value at risk per churned customer: $ 777.14


In [3]:
retention_offer_cost = 20        # assumption: cost of a retention call/discount offer
retention_success_rate = 0.30    # assumption: % of contacted at-risk customers who are actually retained

print("Retention offer cost: $", retention_offer_cost)
print("Assumed retention success rate:", f"{retention_success_rate:.0%}")

Retention offer cost: $ 20
Assumed retention success rate: 30%


In [4]:
import joblib
from sklearn.metrics import confusion_matrix

X_test = pd.read_csv('../data/processed/X_test_fe.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

log_reg = joblib.load('../models/baseline_logistic_regression.pkl')
final_xgb = joblib.load('../models/final_model_xgboost.pkl')

def cost_analysis(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    cost_missed_churners = fn * customer_value_at_risk
    cost_false_alarms = fp * retention_offer_cost
    cost_correct_catches = tp * retention_offer_cost
    value_saved = tp * retention_success_rate * customer_value_at_risk

    net_cost = cost_missed_churners + cost_false_alarms + cost_correct_catches - value_saved

    return {
        'Model': model_name,
        'True Positives (caught)': tp,
        'False Negatives (missed)': fn,
        'False Positives (false alarms)': fp,
        'Cost of missed churners': round(cost_missed_churners, 2),
        'Cost of outreach (all flagged)': round(cost_false_alarms + cost_correct_catches, 2),
        'Value saved by retention': round(value_saved, 2),
        'Net Cost': round(net_cost, 2)
    }

results = [
    cost_analysis(log_reg, X_test, y_test, 'Logistic Regression (baseline)'),
    cost_analysis(final_xgb, X_test, y_test, 'XGBoost (tuned, final)')
]

pd.DataFrame(results)

,Model,True Positives (caught),False Negatives (missed),False Positives (false alarms),Cost of missed churners,Cost of outreach (all flagged),Value saved by retention,Net Cost
0,Logistic Regression (baseline),212,162,113,125896.73,6500,49426.12,82970.61
1,"XGBoost (tuned, final)",296,78,273,60616.94,11380,69010.06,2986.88
